# Task 3 — Viraat's own predictions and evaluation

Run after the planned training finishes. Fixed inputs and the same feature protocol
as the team's reference are retained. Pretrained Inception and AlexNet are used only
for evaluation; submitted images are direct outputs of your own trained CycleGAN.

The uploaded ZIP did **not** include `Part3_Evaluation_Script.ipynb`. Local metrics
therefore remain a reproduction until checked with the unchanged instructor notebook.
No official leaderboard result or real human ratings are invented here.


In [1]:
from pathlib import Path
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import zipfile
import torch

# Optional: set this to the folder containing task3_gan. Otherwise locate it.
PROJECT_ROOT = ""
member_relative = Path("task3_gan/Viraat_Chaudhary")
search_bases = [Path("/app"), Path.cwd(), Path("/content")]
candidates = []
if PROJECT_ROOT:
    candidates.append(Path(PROJECT_ROOT))
else:
    for base in search_bases:
        if not base.is_dir():
            continue
        candidates.append(base)
        candidates.extend(p for p in base.iterdir() if p.is_dir() and not p.name.startswith('.'))
REPO_ROOT = next((p.resolve() for p in candidates if (p/member_relative/'src/train.py').is_file()), None)
if REPO_ROOT is None:
    print("Select the small Viraat_Task3_Ready.zip from this chat.")
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Upload only Viraat_Task3_Ready.zip here; the dataset comes next.")
    archive_name, archive_bytes = next(iter(uploaded.items()))
    import io
    REPO_ROOT = Path('/app') if Path('/app').is_dir() else Path.cwd()
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as z:
        for entry in z.infolist():
            if entry.is_dir():
                continue
            relative = Path(entry.filename)
            if relative.is_absolute() or '..' in relative.parts:
                raise ValueError("Unsafe archive path")
            if not (entry.filename.startswith('task3_gan/Viraat_Chaudhary/') or entry.filename=='START_HERE.md'):
                raise ValueError("This is not the Viraat training ZIP")
            target=REPO_ROOT/relative
            if target.exists():
                raise FileExistsError("Refusing to overwrite existing code: "+str(target))
            z.extract(entry,REPO_ROOT)
    if not (REPO_ROOT/member_relative/'src/train.py').is_file():
        raise RuntimeError("Training files were not found after extracting the ZIP")
    del uploaded,archive_bytes
MEMBER_ROOT = REPO_ROOT/member_relative
SRC = MEMBER_ROOT/'src'
CONFIG = MEMBER_ROOT/'configs/cyclegan_viraat.json'
RUN_ID = "viraat_resizeconv6_run001"
print("Project:",REPO_ROOT)
print("PyTorch:",torch.__version__,"CUDA runtime:",torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable. Reconnect Colab to the lab's Docker local runtime.")
print("GPU:",torch.cuda.get_device_name(0))
print("BF16 supported:",torch.cuda.is_bf16_supported())
print("Run:",RUN_ID)
print("Config:",json.loads(CONFIG.read_text())['training'])


Project: /app/Viraat_Task3_Ready
PyTorch: 2.1.2 CUDA runtime: 12.1
GPU: NVIDIA GeForce RTX 4090
BF16 supported: True
Run: viraat_resizeconv6_run001
Config: {'batch_size': 2, 'beta1': 0.5, 'beta2': 0.999, 'constant_lr_epochs': 20, 'epochs': 60, 'gradient_clip_norm': None, 'image_pool_size': 50, 'learning_rate': 0.0002, 'linear_decay_epochs': 40, 'local_laptop_role': 'development_and_smoke_testing_only', 'mixed_precision': True, 'optimizer': 'adam', 'serious_training_device': 'cuda', 'weight_decay': 0.0, 'precision_dtype': 'bfloat16'}


## Install missing evaluation packages without replacing the CUDA PyTorch build


In [2]:
needed=[]
for module,package in [('lpips','lpips==0.1.4'),('matplotlib','matplotlib>=3.6,<4')]:
    if importlib.util.find_spec(module) is None:
        needed.append(package)
if needed:
    subprocess.run([sys.executable,'-m','pip','install',*needed],check=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.0/125.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 12.3 MB/s eta 0:00:00


## Generate fixed predictions from the completed run


In [3]:
predictions=MEMBER_ROOT/'outputs'/RUN_ID/'predictions'
if not predictions.exists():
    subprocess.run([sys.executable,'-u',str(SRC/'infer.py'),'--run-id',RUN_ID],cwd=REPO_ROOT,check=True)
else:
    print('Predictions already exist; they will be verified during evaluation.')


A2B 50 / 300
A2B 100 / 300
A2B 150 / 300
A2B 200 / 300
A2B 250 / 300
A2B 300 / 300
B2A 50 / 300
B2A 100 / 300
B2A 150 / 300
B2A 200 / 300
B2A 250 / 300
B2A 300 / 300
Predictions: /app/Viraat_Task3_Ready/task3_gan/Viraat_Chaudhary/outputs/viraat_resizeconv6_run001/predictions
Unchanged instructor evaluator working directory: /app/Viraat_Task3_Ready/task3_gan/Viraat_Chaudhary/outputs/viraat_resizeconv6_run001/evaluator_workspace


## Compute all automatic metrics, curves and real-rater audit forms

FID uses a numerically stable sample-space computation of the same sample-covariance
formula. The class-specific “MiFID” is the fixed-index cosine distance in the supplied
team reference, not the general memorization-informed FID definition. LPIPS measures
cycle reconstruction, matching the team's baseline. Human scores and Kaggle fields
remain explicitly pending until real evidence is supplied.


In [4]:
summary=MEMBER_ROOT/'outputs'/RUN_ID/'metrics/evaluation_summary.json'
if not summary.exists():
    subprocess.run([sys.executable,'-u',str(SRC/'evaluate_local.py'),'--run-id',RUN_ID],cwd=REPO_ROOT,check=True)
else:
    print(summary.read_text())


Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth
100%|██████████| 104M/104M [00:08<00:00, 13.0MB/s]


Extracting real_a
Extracting real_b
Extracting pred_A2B
Extracting pred_B2A
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
100%|██████████| 233M/233M [00:18<00:00, 13.3MB/s]


Loading model from: /opt/conda/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth
Cycle metrics: 100 /600
Cycle metrics: 200 /600
Cycle metrics: 300 /600
Cycle metrics: 400 /600
Cycle metrics: 500 /600
Cycle metrics: 600 /600
{
  "run_id": "viraat_resizeconv6_run001",
  "epoch": 60,
  "checkpoint_sha256": "138f9ae9edef412125ddfad643d2fc672c3de2d609819caf0b623979d3765252",
  "protocol": "team reference: first 300 sorted images, Inception-v3 ImageNet features, fixed ordering",
  "directions": {
    "A2B": {
      "fid": 103.03648895627941,
      "class_mifid": 0.41614150006151296,
      "kid_mean": 0.024614788881728068,
      "kid_std": 0.0034065122173219174,
      "generative_precision": 0.6233333333333333,
      "generative_recall": 0.4266666666666667,
      "content_preservation_cosine_similarity": 0.7891364810217774,
      "cycle_l1_rgb01": 0.04343768773600459,
      "cycle_lpips_alex": 0.42009864091873167
    },
    "B2A": {
      "fid": 95.6152380507273,
      "class_mifid": 

## Complete the official evaluator and human audit

Run the **unchanged instructor evaluator** with its working directory set to the
printed private workspace below. Its expected relative `Part 3/Data` folders contain
exact copies of your inputs and direct predictions. Preserve its source and output;
store its official `submission.csv` in your member folder after verifying it.

Give the 30 anonymized panels and separate blank score CSVs to two real raters.
Do not reveal the author/checkpoint or provide `private_manifest.csv`. After both
forms have real scores, run the commented command below. Document your own visual
observations, failure cases and justifications in `results.md` and `failure_analysis.md`.


In [5]:
workspace=MEMBER_ROOT/'outputs'/RUN_ID/'evaluator_workspace'
print('Instructor evaluator working directory:',workspace)
print('Real-rater forms:',MEMBER_ROOT/'outputs'/RUN_ID/'human_audit')
# Run after both real raters fill their CSVs:
# subprocess.run([sys.executable,'-u',str(SRC/'evaluate_local.py'),'--run-id',RUN_ID,'--human-only'],check=True)


Instructor evaluator working directory: /app/Viraat_Task3_Ready/task3_gan/Viraat_Chaudhary/outputs/viraat_resizeconv6_run001/evaluator_workspace
Real-rater forms: /app/Viraat_Task3_Ready/task3_gan/Viraat_Chaudhary/outputs/viraat_resizeconv6_run001/human_audit


## Export your code, outputs, raw logs and latest checkpoint after the run


In [6]:
subprocess.run([sys.executable,'-u',str(SRC/'lab_tools.py'),'export','--run-id',RUN_ID],cwd=REPO_ROOT,check=True)
try:
    from google.colab import files
    files.download(str(REPO_ROOT/'Viraat_Task3_Run_Results.zip'))
except ImportError:
    print('The exported ZIP is in the mounted Windows project folder:',REPO_ROOT/'Viraat_Task3_Run_Results.zip')


Export: /app/Viraat_Task3_Ready/Viraat_Task3_Run_Results.zip | MiB: 412.85
The exported ZIP is in the mounted Windows project folder: /app/Viraat_Task3_Ready/Viraat_Task3_Run_Results.zip
